# VITASA_Enhanced — Training on Google Colab
**Đề tài:** Enhancing Vietnamese Targeted Aspect Sentiment Analysis with Social Media Text Normalization and Imbalanced Learning

**Thứ tự chạy:** Cell 1 → 2 → 3 → 4 (test) → 5 (full training) → 6 (download results)

In [ ]:
# ── Cell 1: Kiểm tra GPU ──────────────────────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name   :', torch.cuda.get_device_name(0))
    print('VRAM       :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠️  Không có GPU — vào Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Clone repo + install dependencies ─────────────────────────────────
!git clone https://github.com/Hunganh1305/VITASA_Enhanced.git
%cd VITASA_Enhanced
!pip install -r requirements.txt -q
print('\n✅ Setup done')

In [ ]:
# ── Cell 3: Verify project ────────────────────────────────────────────────────
import subprocess

# Kiểm tra cấu trúc
result = subprocess.run(['find', '.', '-type', 'f', '-name', '*.py', '-o', '-name', '*.jsonl'],
                        capture_output=True, text=True)
print('Project files:')
for f in sorted(result.stdout.strip().split('\n')):
    print(' ', f)

# Chạy unit tests
print('\nRunning tests...')
!python -m pytest text_normalization/tests/ imbalanced_learning/tests/ -q 2>&1 | tail -3

In [ ]:
# ── Cell 4: Test nhanh 1 config (2 epoch) — xác nhận pipeline OK ─────────────
# Chạy cell này trước, nếu không lỗi thì mới chạy Cell 5 (full)
!python train.py --domain mobile --loss ce --epochs 2
print('\n✅ Test run OK — sẵn sàng chạy full')

In [ ]:
# ── Cell 5: Full training — 4 configs × 3 domains = 12 runs ──────────────────
# ⏱ Ước tính: 6-10 giờ trên T4 GPU (để chạy qua đêm)
import subprocess
import time
import json
from pathlib import Path

configs = [
    # (domain, loss, normalize, label)
    ('mobile',     'ce',    False, 'C1 — Baseline'),
    ('mobile',     'ce',    True,  'C2 — + Text Norm'),
    ('mobile',     'focal', False, 'C3 — + Focal Loss'),
    ('mobile',     'focal', True,  'C4 — + Both'),
    ('restaurant', 'ce',    False, 'C1 — Baseline'),
    ('restaurant', 'ce',    True,  'C2 — + Text Norm'),
    ('restaurant', 'focal', False, 'C3 — + Focal Loss'),
    ('restaurant', 'focal', True,  'C4 — + Both'),
    ('hotel',      'ce',    False, 'C1 — Baseline'),
    ('hotel',      'ce',    True,  'C2 — + Text Norm'),
    ('hotel',      'focal', False, 'C3 — + Focal Loss'),
    ('hotel',      'focal', True,  'C4 — + Both'),
]

EPOCHS = 10
overall_start = time.time()

for i, (domain, loss, normalize, label) in enumerate(configs, 1):
    norm_flag = '--normalize' if normalize else ''
    cmd = f'python train.py --domain {domain} --loss {loss} {norm_flag} --epochs {EPOCHS}'

    print(f'\n{"+"*70}')
    print(f'[{i:02d}/12] {domain.upper()} | {label}')
    print(f'CMD: {cmd}')
    print(f'{"+"*70}')

    t0 = time.time()
    subprocess.run(cmd.split())
    elapsed = time.time() - t0
    print(f'⏱ Done in {elapsed/60:.1f} min')

total = time.time() - overall_start
print(f'\n{"="*70}')
print(f'✅ All 12 configs done in {total/3600:.1f} hours')
print(f'{"="*70}')

In [ ]:
# ── Cell 6: Tổng hợp kết quả thành bảng so sánh ─────────────────────────────
import json
from pathlib import Path

results_dir = Path('experiments/results')
rows = []

for result_file in sorted(results_dir.glob('*/results.json')):
    with open(result_file) as f:
        d = json.load(f)
    rows.append({
        'domain'    : d['domain'],
        'config'    : ('C4' if d['normalize'] and d['loss'] == 'focal' else
                       'C3' if not d['normalize'] and d['loss'] == 'focal' else
                       'C2' if d['normalize'] else 'C1'),
        'normalize' : '✅' if d['normalize'] else '❌',
        'loss'      : d['loss'],
        'dev_f1'    : round(d['best_dev_f1'] * 100, 2),
        'test_f1'   : round(d['test_f1'] * 100, 2),
    })

# In bảng
print(f'{"-"*72}')
print(f'{"Domain":<12} {"Config":<6} {"Norm":<6} {"Loss":<12} {"Dev F1":>8} {"Test F1":>8}')
print(f'{"-"*72}')
for r in sorted(rows, key=lambda x: (x['domain'], x['config'])):
    print(f'{r["domain"]:<12} {r["config"]:<6} {r["normalize"]:<6} {r["loss"]:<12} {r["dev_f1"]:>7.2f}% {r["test_f1"]:>7.2f}%')
print(f'{"-"*72}')

In [ ]:
# ── Cell 7: Download kết quả về máy ──────────────────────────────────────────
# Chỉ zip results.json (không zip best_model.pt vì nặng ~400MB)
!find experiments/results -name 'results.json' | zip results_summary.zip -@

from google.colab import files
files.download('results_summary.zip')
print('✅ Download started')